# Notebook 00 — LangGraph in 20 Minutes

**Workshop:** Agentic AI for Scientists — Week 3 (From Loops to Graphs)
**Block:** core (17 min)
**Goal:** See that a LangGraph graph is nothing more than **state in → node → node → state out**, with edges as the control flow you used to write as `if` and `while`. We build three graphs: one with no LLM at all, one with a *cycle* and a *conditional edge*, and one minimal LLM chatbot — the shape every later notebook reuses.

**The one-sentence version of LangGraph:** in Week 2 the agent *was* a `while` loop over a string scratchpad. That loop can't pause, can't branch cleanly, and forgets everything when the cell ends. LangGraph is the same loop re-expressed as a **graph** — nodes (steps), edges (control flow), and a typed **State** threaded through — so it becomes inspectable, resumable, and persistent.

> **Version note.** Week 2 pinned *classic* LangChain 0.3.x (for `AgentExecutor`). Week 3 is the cutover to the **LangChain / LangGraph 1.0** line — the modern graph API. Same free **Gemini** provider.


## 0. Setup

In [ ]:
%pip install -q \
    "langgraph>=1.2.4" "langchain>=1.3.2" "langchain-core>=1.4.0" \
    "langchain-google-genai>=4.2.4" python-dotenv

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab: add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local: put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# Accept GEMINI_API_KEY as an alias (some AI Studio users store it under that name).
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("API key loaded.")


---

## 1. State — the data the graph passes around

A LangGraph graph has one shared **State**: a typed dict that every node reads from and writes to. Define it once. Here the state is just a string of `text` and an integer `count`.

A node returns a **partial** update — only the keys it changes. LangGraph merges that into the running state. (Later we'll see *reducers* that append instead of overwrite — that's how message history accumulates.)


In [ ]:
from typing import TypedDict


class State(TypedDict):
    text: str
    count: int


# A node is just a function: state in -> partial state out. No LangGraph magic.
def shout(state: State) -> dict:
    return {"text": state["text"].upper()}


def tag(state: State) -> dict:
    return {"text": state["text"] + "  [reviewed]", "count": state["count"] + 1}


print(shout({"text": "hello", "count": 0}))   # {'text': 'HELLO'}
print(tag({"text": "HELLO", "count": 0}))      # {'text': 'HELLO  [reviewed]', 'count': 1}


Those are plain Python functions — you could call them yourself in sequence. The *graph* just formalises "call `shout`, then `tag`" as wiring you can inspect, visualise, pause, and persist.

## 2. Build the graph — nodes + edges

`StateGraph(State)` is the builder. `add_node` registers a function; `add_edge` says "after A, go to B". `START` and `END` are the two sentinels. `compile()` turns the builder into a runnable app you call with `.invoke()`.


In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(State)
builder.add_node("shout", shout)
builder.add_node("tag", tag)

builder.add_edge(START, "shout")    # entry
builder.add_edge("shout", "tag")    # shout -> tag
builder.add_edge("tag", END)        # exit

app = builder.compile()

result = app.invoke({"text": "hello world", "count": 0})
print(result)   # {'text': 'HELLO WORLD  [reviewed]', 'count': 1}


That's a graph: `START -> shout -> tag -> END`. Linear so far — exactly a two-line script. The payoff comes when control flow stops being a straight line.

## 3. The leap a `while` loop can't make cleanly — a *conditional edge* + a *cycle*

Here's the move that justifies the whole framework. A **conditional edge** runs a small router function on the current state and **chooses the next node by name**. Point a node back at an earlier one and you have a controlled **cycle** — a loop whose continue/stop decision lives in the graph, not buried in Python.

This graph keeps reviewing until `count` reaches 3, then stops. No LLM yet — just the control flow.


In [ ]:
def review(state: State) -> dict:
    # one round of "work": bump the counter, annotate the text
    return {"text": state["text"] + f" .{state['count'] + 1}", "count": state["count"] + 1}


def should_continue(state: State) -> str:
    # the router: returns the NAME of the next node (or END)
    if state["count"] >= 3:
        return "stop"
    return "again"


loop = StateGraph(State)
loop.add_node("review", review)
loop.add_edge(START, "review")

# after 'review', run should_continue(state); map its return value -> a destination
loop.add_conditional_edges(
    "review",
    should_continue,
    {"again": "review",   # loop back
     "stop": END},          # exit
)

loop_app = loop.compile()
print(loop_app.invoke({"text": "draft", "count": 0}))
# count climbs 1 -> 2 -> 3, then the router sends it to END


Read that again: `review` points **back to itself** through a conditional edge. That cycle is the spine of every agent we build this week — Reflexion loops generate→reflect→revise, Agentic RAG loops retrieve→grade→maybe-search. The *graph* owns the stop condition, so we can later pause it, checkpoint it, or hand control to a human mid-loop. A bare `while` can do none of that without rewrites.

## 4. Now with an LLM — the minimal chatbot graph (the shape you'll reuse)

Real agents carry a **message history**, not one string. LangGraph's `add_messages` reducer is an annotation on the state field that says "append, don't overwrite" — so each node's output is added to the conversation. This three-line graph is the template under Notebooks 01–04.


In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI


class ChatState(TypedDict):
    # Annotated[..., add_messages] => the reducer APPENDS each node's messages
    messages: Annotated[list, add_messages]


llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


def chatbot(state: ChatState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


g = StateGraph(ChatState)
g.add_node("chatbot", chatbot)
g.add_edge(START, "chatbot")
g.add_edge("chatbot", END)
chat_app = g.compile()

out = chat_app.invoke({"messages": [("user", "Name one reason a scientist would use an agent. One sentence.")]})
print(out["messages"][-1].content)


## 5. Watch it run, step by step — `.stream()`

`.invoke()` gives the final state. `.stream()` yields after **each node**, so you can watch the graph progress — the live equivalent of a trace. Every later notebook uses this to make the agent's reasoning visible.


In [ ]:
for step in loop_app.stream({"text": "draft", "count": 0}):
    print(step)   # {node_name: partial_state_update} after each node fires


**Optional — see the graph as a picture.** A compiled app can render its own topology (handy for the slides and for debugging which edge fired).


In [ ]:
try:
    from IPython.display import Image, display
    display(Image(loop_app.get_graph().draw_mermaid_png()))
except Exception as e:
    # draw_mermaid_png needs network/graphviz; the ASCII fallback always works
    print(loop_app.get_graph().draw_ascii())


---

## Recap

- A **node** is a function: `state -> partial state`. No magic.
- **Edges** are control flow. `add_edge` is a straight line; `add_conditional_edges` is `if`/routing; an edge pointing back is a `while`.
- **State** is one typed dict; `Annotated[list, add_messages]` makes a field *accumulate* (message history).
- `compile()` -> `.invoke()` / `.stream()`.

Everything in Notebooks 01–04 is this, with more nodes and a real reason to branch. Next: **Reflexion** — a generate → reflect → revise cycle that measurably improves a paper abstract.
